# 03 · Tiro a puerta

**Alcance:** experimento tabular reproducible sobre un modelo cinemático acotado. Las dinámicas son aproximaciones didácticas; las curvas que produzca este notebook pertenecen al modelo local. La conexión con `rcssserver` se revisa por separado cuando Docker esté disponible.

Consulta `../docs/goal_shooting.md` para el MDP, las simplificaciones y los criterios de éxito.

## 1. Estado, acciones y recompensa

Cada episodio contiene **un disparo**. El estado discretiza distancia al arco, ángulos a ambos postes y posición del portero (3×4×4×4 = 192 combinaciones). Cada acción fija una potencia y una dirección de KICK. El modelo calcula el cruce de la trayectoria con el arco y la cercanía vertical del portero. Recompensas: +100 por gol, −30 por fallo y −50 por bloqueo. Se estudian por separado arco abierto y portero activo.

`reset()` inicia un episodio y `step(acción)` devuelve `(estado, recompensa, terminado, información)`. El estado contiene índices discretos; `información["success"]` identifica el criterio de éxito.

In [ ]:
import sys
from pathlib import Path

# Funciona con el kernel del contenedor y con un kernel local de VS Code.
for candidate in (
    Path("/workspace/src"),
    Path.cwd() / "src",
    Path.cwd().parent / "src",
):
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "No se encontró src/. Abre el notebook desde el repositorio o notebooks/."
    )

In [ ]:
from metrics import summarize
from plotting import plot_training_comparison, plot_trajectory, plot_value_policy
from tabular import greedy_episode, train_q_learning
from tasks.goal_shooting import GoalShootingEnv

## 2. Entrenamiento y exploración

Se entrena Q-Learning por separado con **arco abierto** y **portero activo**. En cada caso se comparan ε constante y ε decreciente geométrico. Las semillas y parámetros quedan visibles para reproducir los experimentos. Cada episodio es un único disparo.

In [ ]:
EPISODES = 1200
GAMMA = 0.99
ALPHA = 0.2
SEED = 42

results = {}
for scenario, goalie_active in (("Arco abierto", False), ("Portero activo", True)):
    results[scenario] = {}
    for label, mode, epsilon in (
        ("ε fijo", "constant", 0.2),
        ("ε decreciente", "decay", 1.0),
    ):
        env = GoalShootingEnv(seed=SEED, goalie_active=goalie_active)
        q, history = train_q_learning(
            env,
            episodes=EPISODES,
            alpha=ALPHA,
            gamma=GAMMA,
            epsilon_mode=mode,
            epsilon_start=epsilon,
            seed=SEED,
        )
        results[scenario][label] = (q, history)

## 3. Curvas y métricas

`G₀` suma recompensas con descuento. La tasa de éxito cuenta goles por episodio; pasos vale 1 por definición de este MDP. Los resúmenes usan los últimos 100 episodios. Las curvas solo aparecerán al ejecutar el notebook.

In [ ]:
for scenario, runs in results.items():
    print(scenario)
    for label, (_, history) in runs.items():
        print(label, summarize(history))
    plot_training_comparison({label: history for label, (_, history) in runs.items()})

## 4. Valor, política y trayectoria

El mapa muestra un corte de dos dimensiones del estado. Las demás se fijan en la combinación más frecuente entre estados visitados. Las celdas blancas nunca fueron visitadas; el valor y la política son **estimaciones** derivadas de Q.

In [ ]:
q_goalie = results["Portero activo"]["ε decreciente"][0]
plot_value_policy(q_goalie, GoalShootingEnv.state_shape, GoalShootingEnv.action_names)
evaluation_env = GoalShootingEnv(seed=2026, goalie_active=True)
print(greedy_episode(evaluation_env, q_goalie))
plot_trajectory(evaluation_env)

## 5. Conexión con RoboCup

Este notebook entrena en un entorno reducido. El cliente actual todavía no proporciona todas las observaciones necesarias para reproducir esta tarea dentro de `rcssserver` (por ejemplo, portero, postes, compañero o defensor según corresponda). La prueba de conexión UDP está en `01_ball_pursuit.ipynb`; los resultados de este notebook deben identificarse como simulados.